In [ ]:
import os
import numpy as np
import pandas as pd
import fasttext
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import import_ipynb
from notebooks import DataPreProcessing as dp

In [ ]:
# Get preprocessed data from DataPreProcessing notebook
X_train = dp.X_train
X_test = dp.X_test
y_train = dp.y_train
y_test = dp.y_test
clean_text = dp.clean_text
df = dp.df

In [ ]:
# ---------- PATHS ----------
SAVE_DIR = os.path.join(os.getcwd(), 'models')
SAVE_PATH = os.path.join(SAVE_DIR, 'fasttext_model.bin')
TRAIN_TXT = os.path.join(SAVE_DIR, 'fasttext_train.txt')
TEST_TXT = os.path.join(SAVE_DIR, 'fasttext_test.txt')

In [ ]:
def _label_from_int(val):
    """Map 0/1 back to ham/spam for fasttext __label__ format."""
    return 'ham' if val == 0 else 'spam'

def _get_save_paths(dataset_choice='SMS'):
    dataset_key = dataset_choice.lower()
    return {
        'model': os.path.join(SAVE_DIR, f'{dataset_key}_fasttext_model.bin'),
        'train': os.path.join(SAVE_DIR, f'{dataset_key}_fasttext_train.txt'),
        'test': os.path.join(SAVE_DIR, f'{dataset_key}_fasttext_test.txt')
    }

# Update save_fasttext_format function
def save_fasttext_format(X, y, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for text, label in zip(X, y):
            f.write(f'__label__{_label_from_int(label)} {text}\n')

In [ ]:
# ---------- SAVE / LOAD ----------
def save_model(model, dataset_choice='SMS'):
    os.makedirs(SAVE_DIR, exist_ok=True)
    save_path = _get_save_paths(dataset_choice)['model']
    model.save_model(save_path)
    print(f'Model saved to {save_path}')

# Update load_model function
def load_model(dataset_choice='SMS'):
    save_path = _get_save_paths(dataset_choice)['model']
    model = fasttext.load_model(save_path)
    return model

# Update has_saved_model function
def has_saved_model(dataset_choice='SMS'):
    return os.path.exists(_get_save_paths(dataset_choice)['model'])

In [ ]:
# ---------- TRAIN ----------
def train(X_train, y_train, X_test, y_test, dataset_choice='SMS'):
    os.makedirs(SAVE_DIR, exist_ok=True)
    paths = _get_save_paths(dataset_choice)
    
    # Write train/test in fasttext format
    save_fasttext_format(X_train, y_train, paths['train'])
    save_fasttext_format(X_test, y_test, paths['test'])
    
    # Train
    model = fasttext.train_supervised(
        input=paths['train'],
        epoch=25,
        lr=1.0,
        wordNgrams=2
    )
    
    # Evaluate
    result = model.test(paths['test'])
    print(f'Samples: {result[0]}, Precision: {result[1]:.4f}, Recall: {result[2]:.4f}')
    
    # Detailed metrics
    y_true = []
    y_pred = []
    with open(paths['test'], 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(' ', 1)
            if len(parts) == 2:
                true_label = parts[0].replace('__label__', '')
                pred_label = model.predict(parts[1])[0][0].replace('__label__', '')
                y_true.append(true_label)
                y_pred.append(pred_label)
    
    acc = accuracy_score(y_true, y_pred)
    print(f'Accuracy: {acc*100:.2f}%')
    
    # Save model
    save_model(model, dataset_choice)
    
    return model, acc

In [ ]:
# ---------- PREDICT ----------
def predict_message(message, model=None):
    """Predict whether a message is Spam or Ham.
    If model is not passed, loads the saved model automatically."""
    if model is None:
        model = load_model()
    cleaned = clean_text(message)
    prediction = model.predict(cleaned)
    label_raw = prediction[0][0].replace('__label__', '')
    confidence = prediction[1][0] * 100
    label = 'Spam' if label_raw == 'spam' else 'Ham'
    return label, confidence